# Логистическая регрессия

## Почему линейная регрессия не подойдет для классификации

Линейная регрессия может прогнозировать по некоторому тренду следюущие значения в виде определенных значений

В линейной регрессии:
$$
y = w^T x + b
$$

Но если задача **классификации (0/1)** — такая модель плоха:

* может давать значения < 0 или > 1
* не интерпретируется как вероятность


## Основная формула

$$ \sigma(z) = \frac{1}{1 + e^{-z}} $$

Где:
$$
z = w^T x + b
$$

Итог:
$$
P(y=1|x) = \sigma(w^T x + b)
$$

Таким образом сигмоида:
* сжимает значения в диапазон (0,1)
* превращает линейную модель в **вероятностную**

Интерпретация:

* 0.8 → высокая вероятность класса 1
* 0.2 → скорее класс 0

## Логиты (ключевая идея)

Логистическая регрессия моделирует **логарифм шансов (odds)**:

$$
\log \frac{p}{1-p} = w^T x + b
$$

Это называется **logit**

## Функция потерь (вместо MSE)

Используется **log loss / cross-entropy**:

$$
L = - \left[ y \log p + (1-y)\log(1-p) \right]
$$

где y — правильный ответ (0 или 1), а p — предсказанная вероятность.

Почему не MSE:

* градиенты лучше ведут себя
* быстрее и стабильнее сходится

Если применить MSE к логистической регрессии, функция получится «выпукло-вогнутой» (невыпуклой) с кучей локальных минимумов. Градиентный спуск в ней просто «застрянет». Log Loss же всегда выпуклая, что гарантирует нахождение глобального минимума — идеальных весов модели.

## Откуда появляются градиенты

У нас есть три составляющие:

   1. Предсказание (сигмоида): $p = \sigma(z) = \frac{1}{1 + e^{-z}}$, где $z = w \cdot x$ (линейная комбинация).
   2. Функция потерь (Log Loss): $L = -(y \ln(p) + (1 - y) \ln(1 - p))$.
   3. Производная сигмоиды: Важный нюанс — $\frac{dp}{dz} = p(1 - p)$. Это свойство сильно упростит нам жизнь.

2. Применяем цепное правило (Chain Rule)
Чтобы найти, как ошибка меняется в зависимости от весов $w$, нам нужно пройти цепочку:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial p} \cdot \frac{\partial p}{\partial z} \cdot \frac{\partial z}{\partial w}$$

Почему применяется именно цепное правило?

Наша ошибка $L$ зависит не от весов напрямую, а через цепочку событий:
- Изменили веса $w$ изменилось число $z$.
- Изменилось $z$ изменилась вероятность $p$ (сигмоида).
- Изменилась вероятность $p$ изменилась ошибка $L$.
Поэтому, чтобы понять, как «дерганье» веса $w$ отразится на ошибке $L$, мы вынуждены перемножить все эти промежуточные влияния.

3. Считаем по частям:

* Шаг А: Производная функции потерь по предсказанию $p$:

$$\frac{\partial L}{\partial p} = -\left( \frac{y}{p} - \frac{1 - y}{1 - p} \right) = \frac{p - y}{p(1 - p)}$$

* Шаг Б: Производная предсказания по $z$ (та самая сигмоида):

$$\frac{\partial p}{\partial z} = p(1 - p)$$

* Шаг В: Производная $z$ по весам $w$:

$$\frac{\partial z}{\partial w} = x$$

4. Собираем всё вместе:
Перемножаем результаты:

$$\frac{\partial L}{\partial w} = \underbrace{\frac{p - y}{p(1 - p)}}_{\text{Шаг А}} \cdot \underbrace{p(1 - p)}_{\text{Шаг Б}} \cdot \underbrace{x}_{\text{Шаг В}}$$

Знаменатель $p(1 - p)$ и числитель сокращаются и остается чистый результат:

$$\frac{\partial L}{\partial w} = (p - y)x$$



## Градиенты (для понимания обучения)

Очень важный факт:

$$
\frac{\partial L}{\partial w} = (p - y)x
$$

Это почти как в линейной регрессии, но:

* ошибка = (предсказание − истинное значение)
* только предсказание — это вероятность

In [ ]:
import numpy as np

def gradient(X, y, w, b):
    z = X @ w + b
    p = 1 / (1 + np.exp(-z))

    error = p - y  # ключевая строка

    dw = X.T @ error / len(y)
    db = error.mean()

    return dw, db, p, z

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def train_logistic_regression(X, y, lr=0.01, epochs=1000):
    n_samples, n_features = X.shape

    # инициализация
    w = np.zeros(n_features)
    b = 0.0

    for _ in range(epochs):
        dw, db = gradient(X, y, w, b)
        # update
        w -= lr * dw
        b -= lr * db

    return w, b

def log_loss(y, p):
    eps = 1e-15  # защита от log(0)
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def train_logistic_regression_verbose(X, y, lr=0.01, epochs=1000):
    n_samples, n_features = X.shape

    w = np.zeros(n_features)
    b = 0.0

    losses = []

    for epoch in range(epochs):
        dw, db, p, z = gradient(X, y, w, b)
        # update
        w -= lr * dw
        b -= lr * db
        
        loss = log_loss(y, p)
        losses.append(loss)

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

    return w, b, losses

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X=np.array([[0, 0], [0, 1], [1, 0], [1, 1], [0, 0], [0, 1], [1, 0], [1, 1], [0, 0], [0, 1], [1, 0], [1, 1], [0, 0], [0, 1], [1, 0], [1, 1]])
y=np.array([0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

y_pred

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.25


In [ ]:
train_logistic_regression_verbose(X, y, lr=0.01, epochs=1000)

Epoch 0, Loss: 0.6931
Epoch 100, Loss: 0.6931
Epoch 200, Loss: 0.6931
Epoch 300, Loss: 0.6931
Epoch 400, Loss: 0.6931
Epoch 500, Loss: 0.6931
Epoch 600, Loss: 0.6931
Epoch 700, Loss: 0.6931
Epoch 800, Loss: 0.6931
Epoch 900, Loss: 0.6931


(array([0., 0.]),
 0.0,
 [0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931471805599453,
  0.6931

## Порог (threshold)

По умолчанию: в основном для прогнозирования предполагают, что для того, чтобы поставить label = 1 нам нужен порог в 0.5 (т. е. 50 процентов вероятности). Однако иногда это поведение можно менять


Но можно менять на другие значения: если p больше 0.7, то тогда распределяем значения в класс 1, иначе в класс 2

Это критично для данных, в которых надо быть сильно уверенным, например:

* fraud detection
* medical diagnosis

## Регуляризация

Встроена почти всегда:

### L2

$$
\lambda ||w||^2
$$

### L1

$$
\lambda ||w||
$$



## Препроцессинг перед применением модели

**Применение StandartScaler**

Логистическая регрессия обучается с помощью градиентного спуска.
- Если один признак (например, «доход») измеряется в тысячах, а другой («возраст») — в десятках, линии уровня функции потерь превращаются в очень вытянутые эллипсы.
- Градиентный спуск в таких условиях начинает «скакать» из стороны в сторону, и обучение идет очень медленно.
- StandardScaler делает «овраг» круглым, и алгоритм идет к минимуму по кратчайшему пути.

Если данные в одном масштабе, ты можешь посмотреть на коэффициенты модели:
- Чем больше абсолютное значение веса $w$, тем важнее признак.
- Без StandardScaler вес при «возрасте» может быть 0.5, а при «доходе» 0.00001, и ты никогда не поймешь, какой фактор на самом деле сильнее влияет на прогноз.

При наличии категориальных переменных: **One-Hot Encoding**;

Логарифмическая регрессия, как и линейная регрессия, не устойчива к выбросам, что говорит о необходимости их фильтрации;

Как и линейная регрессия в логистической регрессии очень важно удалить мультиколлинеарные признаки (т. к. обучение строится на основе числовых характеристик)